# Fine-tuning Phi-3 Model for Vermeg Chatbot (Colab Version)

This notebook is optimized for running on Google Colab with GPU acceleration. Make sure to:
1. Upload your dataset to Google Drive
2. Select GPU runtime in Colab (Runtime > Change runtime type > GPU)
3. Mount your Google Drive

## 1. Setup and Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q transformers==4.36.0
!pip install -q peft
!pip install -q bitsandbytes
!pip install -q accelerate
!pip install -q datasets

In [ ]:
# Import required libraries
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
import logging
import bitsandbytes as bnb

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 2. Verify GPU Setup

In [ ]:
# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    
# Enable memory efficient optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 3. Chatbot Trainer Class

In [ ]:
class VermegChatbotTrainer:
    def __init__(self, model_name="microsoft/Phi-3-mini-4k-instruct"):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.dataset = None
        
    def load_model_and_tokenizer(self):
        """Load the base model and tokenizer with GPU optimizations"""
        logger.info(f"Loading model: {self.model_name}")
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        # Load model with 8-bit quantization for GPU efficiency
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            trust_remote_code=True,
            device_map="auto",
            load_in_8bit=True,
            torch_dtype=torch.float16
        )
        
        logger.info("Model and tokenizer loaded successfully!")
        
    def setup_lora(self):
        """Set up LoRA for efficient fine-tuning"""
        logger.info("Setting up LoRA configuration...")
        
        # Configure LoRA
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            target_modules=["self_attn.qkv_proj", "self_attn.o_proj", "mlp.gate_up_proj", "mlp.down_proj"],
            bias="none",
            fan_in_fan_out=False,
            init_lora_weights=True
        )
        
        # Apply LoRA
        self.model = get_peft_model(self.model, lora_config)
        self.model.enable_input_require_grads()
        
        # Print trainable parameters
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.model.parameters())
        logger.info(f"Trainable parameters: {trainable_params:,}")
        logger.info(f"Total parameters: {total_params:,}")
        logger.info(f"Percentage trainable: {100 * trainable_params / total_params:.2f}%")
        
    def prepare_dataset(self, data_file):
        """Prepare the dataset for training"""
        logger.info(f"Loading dataset from {data_file}")
        
        with open(data_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        formatted_data = []
        for item in data:
            conversation = f"<|system|>\nYou are a helpful assistant that provides information about Vermeg's financial technology solutions.<|end|>\n<|user|>\n{item['input']}<|end|>\n<|assistant|>\n{item['output']}<|end|>"
            formatted_data.append({"text": conversation})
        
        self.dataset = Dataset.from_list(formatted_data)
        split_dataset = self.dataset.train_test_split(test_size=0.1, seed=42)
        self.train_dataset = split_dataset['train']
        self.eval_dataset = split_dataset['test']
        
        logger.info(f"Training samples: {len(self.train_dataset)}")
        logger.info(f"Validation samples: {len(self.eval_dataset)}")
        
    def tokenize_dataset(self):
        """Tokenize the dataset"""
        logger.info("Tokenizing dataset...")
        
        def tokenize_function(examples):
            return self.tokenizer(
                examples["text"], 
                truncation=True, 
                padding=False, 
                max_length=1024
            )
        
        self.train_dataset = self.train_dataset.map(
            tokenize_function, 
            batched=True,
            remove_columns=self.train_dataset.column_names
        )
        
        self.eval_dataset = self.eval_dataset.map(
            tokenize_function, 
            batched=True,
            remove_columns=self.eval_dataset.column_names
        )
        
        logger.info("Dataset tokenization complete!")
        
    def train(self, output_dir):
        """Train the model with GPU optimizations"""
        logger.info("Starting training...")
        
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=3,
            per_device_train_batch_size=4,  # Increased for GPU
            gradient_accumulation_steps=4,
            learning_rate=5e-5,
            max_grad_norm=0.5,
            warmup_ratio=0.03,
            logging_steps=10,
            save_strategy="steps",
            save_steps=100,
            save_total_limit=3,
            remove_unused_columns=False,
            push_to_hub=False,
            fp16=True,  # Enable mixed precision training
            gradient_checkpointing=True,
            report_to="none"
        )
        
        # Data collator
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False
        )
        
        # Initialize trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            data_collator=data_collator,
        )
        
        # Train and save
        trainer.train()
        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        
        logger.info(f"Training complete! Model saved to {output_dir}")

## 4. Training Pipeline

In [ ]:
# Set paths for data and output
DATA_FILE = "/content/drive/MyDrive/vermeg_dataset_instruction.json"  # Update this path
OUTPUT_DIR = "/content/drive/MyDrive/vermeg-chatbot-finetuned"  # Update this path

# Initialize and train
trainer = VermegChatbotTrainer()
trainer.load_model_and_tokenizer()
trainer.setup_lora()
trainer.prepare_dataset(DATA_FILE)
trainer.tokenize_dataset()
trainer.train(OUTPUT_DIR)

## 5. Test the Model

In [ ]:
def test_model(model, tokenizer, question):
    # Format the input
    prompt = f"<|system|>\nYou are a helpful assistant that provides information about Vermeg's financial technology solutions.<|end|>\n<|user|>\n{question}<|end|>\n<|assistant|>\n"
    
    # Generate response
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract only the assistant's response
    response = response.split("<|assistant|>\n")[1].split("<|end|")[0].strip()
    return response

# Test questions
test_questions = [
    "What is Xchanger?",
    "Tell me about Vermeg's solutions",
    "How can Easy Agreement help my business?",
    "What are the benefits of using Colline?"
]

for question in test_questions:
    print(f"\nQuestion: {question}")
    print(f"Answer: {test_model(trainer.model, trainer.tokenizer, question)}\n")